# Mengambil ulasan aplikasi dari Google Play Store
Notebook ini mengambil ulasan satu aplikasi dan menyimpannya sebagai CSV.

**Cara memakai:** buka notebook di Google Colab, jalankan langkah 1, ubah pengaturan langkah 2, lalu jalankan langkah 3–5 secara berurutan dengan tombol ▶ di sebelah kiri sel. Tidak memerlukan GPU.

Untuk aplikasi lain, cukup ubah `APP_ID` pada langkah 2, lalu jalankan kembali langkah 2–5. Setiap proses dimulai dari awal; hasil tidak digabung dengan proses sebelumnya. Target awal 100 ulasan agar mudah dicoba; ubah menjadi 10.000 setelah pengaturan sesuai.

Jumlah hasil dapat lebih kecil daripada target. Pengaturan bahasa/negara bukan jaminan bahwa seluruh ulasan global atau seluruh penulis dari negara tertentu akan tercakup.

## 1. Pasang paket dan impor alat
Jalankan sekali setiap kali memulai sesi Colab baru. Tunggu hingga selesai sebelum melanjutkan.

In [ ]:
%pip install -q google-play-scraper pandas

import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from google_play_scraper import Sort, reviews
from IPython.display import display

## 2. Pilih aplikasi dan jumlah ulasan
Buka halaman aplikasi di Play Store melalui browser. Salin teks setelah `id=` sampai sebelum `&` (jika ada).

Contoh: `https://play.google.com/store/apps/details?id=com.whatsapp&hl=id` → `com.whatsapp`.

**Ganti `APP_ID` berikut untuk mengambil ulasan aplikasi lain.** Nilai awal memakai ID pada sel terakhir notebook asli; periksa kembali halaman aplikasinya sebelum menjalankan. Jangan masukkan seluruh URL atau nama tampilan aplikasi.

In [ ]:
# ===== PENGATURAN YANG BOLEH DIGANTI =====
APP_ID = "com.capitalx.blissfully"  # GANTI dengan ID aplikasi dari URL Play Store.
JUMLAH_ULASAN = 100              # GANTI target, misalnya 1000 atau 10000.
BAHASA = "id"                    # GANTI ke "en" untuk bahasa Inggris.
NEGARA = "id"                    # GANTI ke "us" untuk wilayah Amerika Serikat.
URUTAN = Sort.NEWEST              # Pilihan lain: Sort.MOST_RELEVANT.

# Biasanya tidak perlu diubah.
UKURAN_BATCH = 200                # Jumlah ulasan yang diminta per pemanggilan.
JEDA_DETIK = 1.0                  # Jeda antarpemanggilan untuk mengurangi laju permintaan.

# Periksa isian agar kesalahan pengaturan mudah dipahami.
if not isinstance(APP_ID, str) or not re.fullmatch(r"[A-Za-z0-9_]+(?:\.[A-Za-z0-9_]+)+", APP_ID):
    raise ValueError("APP_ID harus berupa ID seperti com.whatsapp, bukan URL lengkap.")
if type(JUMLAH_ULASAN) is not int or JUMLAH_ULASAN < 1:
    raise ValueError("JUMLAH_ULASAN harus bilangan bulat lebih besar dari 0.")
if type(UKURAN_BATCH) is not int or not 1 <= UKURAN_BATCH <= 200:
    raise ValueError("UKURAN_BATCH harus bilangan bulat antara 1 dan 200.")
if not isinstance(JEDA_DETIK, (int, float)) or not 0 <= JEDA_DETIK <= 60:
    raise ValueError("JEDA_DETIK harus berupa angka antara 0 dan 60.")
if URUTAN not in (Sort.NEWEST, Sort.MOST_RELEVANT):
    raise ValueError("Gunakan Sort.NEWEST atau Sort.MOST_RELEVANT.")
for label, nilai in [("BAHASA", BAHASA), ("NEGARA", NEGARA)]:
    if not isinstance(nilai, str) or not re.fullmatch(r"[a-z]{2}", nilai):
        raise ValueError(f"{label} harus dua huruf kecil, misalnya id.")

print(f"Aplikasi: {APP_ID} | Target: {JUMLAH_ULASAN:,} ulasan")

## 3. Ambil ulasan
Jalankan sel ini dan tunggu. Progres ditampilkan setiap batch (sekumpulan ulasan).

Kode menghindari ID ulasan berulang, membatasi hasil sesuai target, dan berhenti jika respons kosong, penanda halaman berikutnya habis, atau satu batch tidak menambah ulasan baru. Berhenti karena tidak ada data baru tidak membuktikan semua ulasan sudah terkumpul.

Jika ada error yang diteruskan oleh paket atau Anda menghentikan sel sekali dengan tombol stop, data yang sudah terkumpul tetap dapat disimpan melalui langkah 4. Memutus atau mereset runtime dapat menghilangkan data di memori.

In [ ]:
def ambil_ulasan():
    """Ambil ulasan unik dengan pengaturan langkah 2; kembalikan data dan status."""
    hasil = []
    id_terkumpul = set()
    token = None
    status = "Target tercapai."

    try:
        while len(hasil) < JUMLAH_ULASAN:
            batch, token = reviews(
                APP_ID,
                lang=BAHASA,
                country=NEGARA,
                sort=URUTAN,
                count=min(UKURAN_BATCH, JUMLAH_ULASAN),
                continuation_token=token,
            )
            if not batch:
                status = "Respons kosong; data mungkin habis atau permintaan gagal."
                break

            jumlah_sebelumnya = len(hasil)
            for ulasan in batch:
                review_id = ulasan.get("reviewId")
                if not review_id:
                    raise ValueError("Respons berisi ulasan tanpa reviewId; periksa kompatibilitas paket.")
                if review_id not in id_terkumpul:
                    id_terkumpul.add(review_id)
                    hasil.append(ulasan)
                # Token dapat mempertahankan ukuran batch awal; batasi hasil di sini.
                if len(hasil) >= JUMLAH_ULASAN:
                    break

            print(f"Terkumpul: {len(hasil):,} / {JUMLAH_ULASAN:,} ulasan unik")
            if len(hasil) >= JUMLAH_ULASAN:
                break
            if len(hasil) == jumlah_sebelumnya:
                status = "Batch tidak menambah ulasan baru; proses dihentikan."
                break
            if token is None or getattr(token, "token", None) is None:
                status = "Tidak ada penanda halaman berikutnya; hasil mungkin belum lengkap."
                break
            time.sleep(JEDA_DETIK)
    except KeyboardInterrupt:
        status = "Dihentikan pengguna; hasil parsial dipertahankan."
    except Exception as error:
        status = f"Proses terhenti ({type(error).__name__}): {error}"

    return hasil, status

# Reset hasil setiap kali sel ini dijalankan agar aplikasi tidak tercampur.
file_csv = None
hasil_ulasan, status_proses = ambil_ulasan()
df = pd.DataFrame(hasil_ulasan)
# Rekam pengaturan saat pengambilan, bukan saat pengaturan diedit kemudian.
info_proses = {"app_id": APP_ID, "bahasa": BAHASA, "negara": NEGARA, "urutan": URUTAN.name}
print(f"\n{status_proses}")
print(f"Jumlah akhir: {len(df):,} ulasan. Lanjutkan ke langkah 4.")

## 4. Lihat dan simpan hasil
Semua kolom dari sumber dipertahankan. Kolom penting: `content` (teks ulasan), `score` (bintang), `at` (tanggal), `reviewId` (ID unik), dan `thumbsUpCount` (jumlah tanda membantu).

CSV menggunakan UTF-8 dengan BOM agar karakter Indonesia lebih mudah terbaca di Excel. Nama file otomatis memuat ID aplikasi dan waktu UTC; hasil parsial diberi penanda pada nama. File tersimpan sementara di runtime Colab sampai diunduh. Jika hasil kosong, tidak ada CSV yang dibuat.

In [ ]:
file_csv = None
if df.empty:
    print("Belum ada ulasan. Periksa APP_ID, koneksi, bahasa, negara, dan pesan langkah 3.")
else:
    # Tambahkan asal data agar CSV tetap jelas saat dibuka kemudian.
    for kolom, nilai in info_proses.items():
        df[kolom] = nilai
    df["status_proses"] = status_proses
    penanda = "" if status_proses == "Target tercapai." else "_parsial"
    waktu = datetime.utcnow().strftime("%Y%m%d_%H%M%S_%f")
    file_csv = Path(f"ulasan_{info_proses['app_id']}_{waktu}_UTC{penanda}.csv")
    df.to_csv(file_csv, index=False, encoding="utf-8-sig")
    display(df.head(5))
    print(f"Tersimpan: {file_csv.name} ({len(df):,} ulasan)")

## 5. Unduh CSV ke komputer
Jalankan sel ini setelah langkah 4. Jika unduhan tidak muncul, buka panel **Files / File** di kiri Colab, lalu pilih **Download / Unduh** pada berkas CSV.

In [ ]:
if file_csv is None or not file_csv.exists():
    print("Tidak ada CSV untuk diunduh. Jalankan langkah 3 dan 4 terlebih dahulu.")
else:
    try:
        from google.colab import files
    except ImportError:
        print(f"Di luar Colab, file tersedia di: {file_csv.resolve()}")
    else:
        files.download(str(file_csv))

## Jika hasil tidak sesuai harapan
- **Nama variabel belum dikenal / NameError:** jalankan sel berurutan dari langkah 1.
- **Ulasan kosong atau lebih sedikit dari target:** pastikan ID dan ketersediaan aplikasi di wilayah yang dipilih. Respons kosong juga dapat terjadi saat permintaan gagal; jangan langsung menganggap aplikasi tidak punya ulasan.
- **Ingin aplikasi lain:** ubah `APP_ID`, lalu jalankan langkah 2–5. Nama CSV menyesuaikan otomatis.
- **Ingin hasil berbeda:** ubah bahasa, negara, atau urutan secara sadar. `en`/`us` bukan mode “seluruh dunia”, dan perubahan urutan bukan jaminan menambah jumlah ulasan.
- **Paket atau Google Play berubah:** instalasi yang berhasil belum menjamin scraping berhasil. Paket dapat menangani sebagian error secara internal tanpa meneruskannya ke notebook.

### Catatan pemeriksaan
Notebook asli memiliki tiga alur untuk aplikasi berbeda, impor dan tampilan tabel berulang, serta nama CSV yang tidak konsisten dengan ID aplikasi. Revisi menyatukannya, menambah validasi, deduplikasi ID, batas hasil yang tepat, penanganan hasil kosong/error, dan unduhan.

Validasi revisi dilakukan untuk struktur notebook, sintaks sel Python, dan simulasi respons scraping. Pengambilan langsung dari Google Play serta unduhan browser Colab belum diuji di lingkungan ini.

Referensi teknis: [dokumentasi google-play-scraper](https://github.com/JoMingyu/google-play-scraper) dan [implementasi reviews](https://github.com/JoMingyu/google-play-scraper/blob/master/google_play_scraper/features/reviews.py).